In [1]:
import pandas as pd

In [2]:
df1 = pd.read_excel("Double sort Coefficients 2 (1).xlsx",sheet_name="Coefficients")
df1

,Unnamed: 0,CMA,SMB,RMW,HML,Rmkt,Rriskfree
0,2016-02-29,-0.374327,0.049256,-0.164032,0.480728,-0.079242,-0.0215
1,2016-03-31,-0.375640,0.024494,-0.198931,0.594489,0.109767,-0.3200
2,2016-04-30,-0.367142,-0.010662,-0.197901,0.576125,0.021354,0.4800
3,2016-05-31,-0.369445,0.014043,-0.196493,0.561952,0.034051,-0.3100
4,2016-06-30,-0.368810,0.014543,-0.172801,0.538052,0.028868,-0.0383
...,...,...,...,...,...,...,...
102,2024-12-31,-0.197353,-0.026512,-0.172050,0.468739,-0.013655,-0.0104
103,2025-01-31,-0.288577,-0.008133,-0.138691,0.435803,-0.034591,0.0045
104,2025-02-28,-0.288577,-0.003778,-0.153813,0.504879,-0.077684,-0.0208
105,2025-03-31,-0.215169,-0.025823,-0.129311,0.482106,0.073606,-0.0337


In [5]:
df2 = pd.read_excel("Double sort Coefficients 2 (1).xlsx",sheet_name="Portfolio Returns")
df2

,Unnamed: 0,SMALL-CONSERVATIVE,SMALL-NEUTRAL,SMALL-AGGRESSIVE,BIG-CONSERVATIVE,BIG-NEUTRAL,BIG-AGGRESSIVE,SMALL-HIGH,SMALL-LOW,SMALL-NEUTRAL.1,BIG-HIGH,BIG-NEUTRAL.1,BIG-LOW,SMALL-ROBUST,SMALL-NEUTRAL.2,SMALL-WEAK,BIG-ROBUST,BIG-NEUTRAL.2,BIG-WEAK
0,2016-02-29,-0.118122,-0.053983,-0.077713,-0.061640,-0.071659,-0.076498,-0.109407,0.000000,-0.054476,-0.079639,-0.067939,0.000000,-0.061504,-0.104414,-0.091015,-0.017195,-0.081176,-0.107433
1,2016-03-31,0.095272,0.095969,0.110088,0.099666,0.098168,0.089001,0.101856,0.070563,0.113660,0.128320,0.092338,0.007384,0.075429,0.126703,0.102267,0.073343,0.016034,0.162017
2,2016-04-30,0.014691,0.013868,0.056162,0.036717,0.012920,0.031197,0.021959,0.040760,0.025313,0.046044,0.020205,0.006235,0.050819,0.021297,0.018665,0.004571,0.029452,0.038246
3,2016-05-31,0.024639,-0.031474,0.032776,0.015163,0.043986,0.028378,0.006406,0.021210,-0.012080,0.040837,0.038145,0.002202,0.006547,0.029578,-0.003832,0.017408,0.008325,0.064438
4,2016-06-30,0.072524,0.094242,0.039772,0.059707,0.011767,0.045449,0.091772,0.003245,0.050583,0.033938,0.013943,0.047838,0.058304,0.083767,0.069128,0.036652,0.030305,0.032013
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,2024-12-31,0.011649,-0.004867,0.045718,0.017988,-0.013018,0.026058,0.049325,-0.006092,-0.000929,-0.003816,-0.002126,0.027858,0.021777,-0.002036,0.024210,0.033290,0.004653,0.000282
103,2025-01-31,-0.064377,-0.093339,-0.131358,-0.048011,-0.027057,-0.062881,-0.056260,-0.129027,-0.077978,-0.039953,-0.043691,-0.049920,-0.144126,-0.083306,-0.033168,-0.050607,-0.074375,-0.021551
104,2025-02-28,-0.131335,-0.079460,-0.077117,-0.085019,-0.085199,-0.061037,-0.146736,-0.014662,-0.139702,-0.088846,-0.073919,-0.073943,-0.099515,-0.117389,-0.069669,-0.080504,-0.082224,-0.079616
105,2025-03-31,0.037812,0.094896,0.053169,0.085187,0.068513,0.070078,0.085683,-0.002814,0.060056,0.100994,0.050956,0.074948,0.075996,0.051031,0.059815,0.072223,0.061141,0.082507


In [7]:
df1.fillna(0, inplace=True)

In [9]:
df2.fillna(0, inplace=True)

In [11]:
df1.columns

Index(['Unnamed: 0', 'CMA', 'SMB', 'RMW', 'HML', 'Rmkt', 'Rriskfree'], dtype='object')

In [13]:
df2.columns[1:]

Index(['SMALL-CONSERVATIVE', 'SMALL-NEUTRAL', 'SMALL-AGGRESSIVE',
       'BIG-CONSERVATIVE', 'BIG-NEUTRAL', 'BIG-AGGRESSIVE', 'SMALL-HIGH',
       'SMALL-LOW', 'SMALL-NEUTRAL.1', 'BIG-HIGH', 'BIG-NEUTRAL.1', 'BIG-LOW',
       'SMALL-ROBUST', 'SMALL-NEUTRAL.2', 'SMALL-WEAK', 'BIG-ROBUST',
       'BIG-NEUTRAL.2', 'BIG-WEAK'],
      dtype='object')

In [15]:
import pandas as pd
import statsmodels.api as sm

# List of columns with % signs to clean
cols = ['SMB', 'HML', 'Rriskfree', 'Rmkt']

df = pd.DataFrame()

nse_cols = df2.columns[1:]

results = []
df1['MKT_RF'] = df1['Rmkt'] - df1['Rriskfree']

for col in nse_cols:
    # Compute Excess Returns
    df['Equity'] = df2[col] - df1['Rriskfree']
        
    # Define independent variables (X) and dependent (Y)
    X = df1[['MKT_RF', 'SMB', 'HML']]
    X = sm.add_constant(X)  # Adds alpha (intercept) to model
    y = df['Equity']

    # Run regression
    model = sm.OLS(y, X).fit()

    values = model.params.values
    errors = model.bse.values
    list_val = list(values)
    list_val.insert(0,col)
    #list_val.insert(0,name)
    list_val.extend(errors)
    results.append(list_val)

    print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 Equity   R-squared:                       0.933
Model:                            OLS   Adj. R-squared:                  0.931
Method:                 Least Squares   F-statistic:                     481.1
Date:                Sun, 06 Jul 2025   Prob (F-statistic):           2.03e-60
Time:                        18:50:27   Log-Likelihood:                 183.05
No. Observations:                 107   AIC:                            -358.1
Df Residuals:                     103   BIC:                            -347.4
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0070      0.018      0.393      0.6

In [17]:
results

[['SMALL-CONSERVATIVE',
  0.007023609130203428,
  1.0350845941372684,
  -0.04060128079011134,
  -0.009928052626331688,
  0.01785246348197808,
  0.027323486225615597,
  0.1012010565766573,
  0.029525657260418583],
 ['SMALL-NEUTRAL',
  0.014314493853328468,
  0.9918276412792906,
  -0.10461635111780991,
  -0.023431434996450652,
  0.01515428764673311,
  0.02319388414895944,
  0.08590578678758612,
  0.025063224665600042],
 ['SMALL-AGGRESSIVE',
  0.01002224420847347,
  1.0190391882560281,
  0.06700643845376872,
  0.008308006883091548,
  0.019459627524782506,
  0.029783276978314878,
  0.11031165912111904,
  0.03218369796931095],
 ['BIG-CONSERVATIVE',
  -0.003254013066980261,
  0.986892097526645,
  0.09315753606183133,
  0.014908532066797549,
  0.008822469066616132,
  0.013502932649096541,
  0.050012324184712444,
  0.014591218635708997],
 ['BIG-NEUTRAL',
  -0.010886452602050584,
  1.0010226260134187,
  0.04751430485224452,
  0.02656782937734899,
  0.008426084344841921,
  0.012896259374207866,


In [19]:
output_df = pd.DataFrame(results,columns=["Name","Alpha","Bmkt","Bsmb","Bhml","Ealpha","Ebmkt","Ebsmb","Ebhml"])
output_df.to_excel("output_ff3.xlsx",index=False)

In [21]:
output_xl= pd.read_excel("output_ff3.xlsx")

In [23]:
output_xl.columns

Index(['Name', 'Alpha', 'Bmkt', 'Bsmb', 'Bhml', 'Ealpha', 'Ebmkt', 'Ebsmb',
       'Ebhml'],
      dtype='object')

In [21]:
# Select only numeric columns (exclude 'Data_Section' and 'Index')
numeric_cols = output_xl.drop(columns=['Name'])

# Calculate statistics
mean_vals = numeric_cols.mean()
median_vals = numeric_cols.median()
std_vals = numeric_cols.std()
correlation_matrix = numeric_cols.corr()

# Combine mean, median, std into a summary DataFrame
summary_df = pd.DataFrame({
    'Mean': mean_vals,
    'Median': median_vals,
    'Standard Deviation': std_vals
})

# Save results to Excel
with pd.ExcelWriter("statistical_summary_ff3.xlsx") as writer:
    summary_df.to_excel(writer, sheet_name="Summary Stats")
    correlation_matrix.to_excel(writer, sheet_name="Correlation Matrix")

print("Summary statistics and correlation matrix saved to 'statistical_summary_ff3.xlsx'")

Summary statistics and correlation matrix saved to 'statistical_summary_ff3.xlsx'


In [29]:
import numpy as np
import statsmodels.api as sm
from scipy.stats import f

def calculate_grs_with_pvalue(df, asset_cols, factor_cols, df1,df2,rf_col='Rft', market_col='Rmt',):
    T = len(df)                      # Number of time periods
    N = len(asset_cols)             # Number of assets
    K = len(factor_cols)            # Number of factors

    alphas = []
    residuals = []

    # Step 1: Regress each asset’s excess return on the factors
    for col in asset_cols:
        df[col] = df2[col] - df1[rf_col]
        #df['MKT_RF'] = df1['Rmt'] - df['Rft']

        X = df[factor_cols].copy()
        X = sm.add_constant(X)
        y = df[col]
        model = sm.OLS(y, X).fit()

        alphas.append(model.params[0])           # Intercept (alpha)
        residuals.append(model.resid.values)     # Residuals for this asset

    # Step 2: Prepare GRS statistic components
    alphas = np.array(alphas).reshape(-1, 1)      # Shape: N x 1
    residuals = np.array(residuals).T             # Shape: T x N

    Sigma = np.cov(residuals, rowvar=False)       # Residual covariance matrix: N x N
    Sigma_inv = np.linalg.inv(Sigma)              # Inverse covariance

    factor_mean = df[factor_cols].mean().values.reshape(-1, 1)  # K x 1
    factor_cov = np.cov(df[factor_cols].T)                      # K x K

    f_term = factor_mean.T @ np.linalg.inv(factor_cov) @ factor_mean  # scalar

    grs_numerator = (T / N) * (alphas.T @ Sigma_inv @ alphas)
    grs_denominator = 1 + f_term
    GRS_stat = float(grs_numerator / grs_denominator)

    # Step 3: Compute p-value using F-distribution
    df1 = N
    df2 = T - N - K
    p_value = 1 - f.cdf(GRS_stat, df1, df2)

    return GRS_stat, p_value


In [35]:
# Asset and factor columns
asset_cols = df2.columns[1:]
factor_cols = ['MKT_RF', 'SMB', 'HML','CMA','RMW']

df3 = df1[factor_cols]  # make sure MKT_RF exists
grs, pval = calculate_grs_with_pvalue(df3, asset_cols, factor_cols,df1,df2)
print(f"GRS: {grs:.4f}, p-value: {pval:.4f}")


C:\Users\hp\AppData\Local\Temp\ipykernel_25416\221229629.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df2[col] - df1[rf_col]
C:\Users\hp\AppData\Local\Temp\ipykernel_25416\221229629.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  alphas.append(model.params[0])           # Intercept (alpha)
C:\Users\hp\AppData\Local\Temp\ipykernel_25416\221229629.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: htt

GRS: 8.1174, p-value: nan


C:\Users\hp\AppData\Local\Temp\ipykernel_25416\221229629.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  alphas.append(model.params[0])           # Intercept (alpha)
C:\Users\hp\AppData\Local\Temp\ipykernel_25416\221229629.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = df2[col] - df1[rf_col]
C:\Users\hp\AppData\Local\Temp\ipykernel_25416\221229629.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by